# THOR — Querying a broker: `load()`, `search()`, `cone_search()`, `get()`

<div align="center">
<img src="../figures/logo.svg" width="600">
</div>

**Author:** Fabio Ragosta (they/them), fix-term researcher at the University of Naples "Federico II"
**Tutorial 2 of 3.**

In production, `Hunter.load("fink")` (or `"lasair"`, `"alerce"`) creates a real REST client that talks to
the broker's public API (e.g. `https://api.ztf.fink-portal.org`), so it requires a network connection
(and, for Lasair/ALeRCE, credentials). To make this notebook runnable anywhere without network access,
**we only replace the loaded broker's `search`/`cone_search`/`get_object` method** with a function that
returns `Candidate` objects shaped exactly like the real parsers (`FinkParser`, `LasairParser`, ...)
normally produce.

Every time we do this, a comment shows the real network call being replaced. In your own environment,
with network access available, you can simply skip these override lines.


In [1]:
from thor.model import Candidate, Coordinates, BrokerInfo, Classification
from thor.hunter import Hunter

hunter = Hunter()
hunter.load("fink")   # in production: creates a FinkREST client against the Fink API
print(hunter.broker.name, "loaded. Survey:", hunter.survey)


Fink loaded. Survey: ztf


## 1. `Hunter.search()`

`search()` only queries **the broker loaded with `load()`**, then still routes the results through
`FusionService.run()` (for code uniformity) and finally through `Hunter.process()` (features,
cross-match, calibration, ranking). With a single broker, fusion is effectively a pass-through: it
groups by `object_id`, but every group contains a single broker, so the weight cancels out in the
ratio and the fused classification matches the original one (only relabeled as `"THOR Fusion"`).


In [2]:
def simulated_fink_search(classifier=None, limit=100):
    """
    Replaces FinkBroker.search(), which in production calls:
        POST https://api.ztf.fink-portal.org/api/v1/latests
    """
    c = Candidate(coordinates=Coordinates(ra=150.324, dec=2.211))
    c.add_broker(BrokerInfo(broker="Fink", object_id="ZTF23aabbcc"))
    c.classification = Classification(
        probabilities={"SN Ia": 0.74, "SN II": 0.20, "AGN": 0.06},
        classifier=classifier or "SuperNNova",
    )
    for mjd, mag in [(60001.2, 19.4), (60003.5, 18.9), (60006.1, 18.7)]:
        c.add_detection(mjd=mjd, mag=mag, magerr=0.05, filt="g")
    return [c]

hunter.broker.search = simulated_fink_search  # tutorial-only override

results = hunter.search(classifier="SN")

for r in results:
    print(r)
    print("  fused classifier:", r.classification.classifier)
    print("  total ranking:", r.ranking.total)


Candidate(object='ZTF23aabbcc', class='SN Ia (74.00%)', RA=150.32400, Dec=2.21100, brokers=[Fink])
  fused classifier: THOR Fusion
  total ranking: 0.5705


## 2. `Hunter.cone_search()` — a known bug

`cone_search()` is meant to do the same thing as `search()`, but for a cone search around coordinates.
In the current code, however, it calls `self.fusion(candidates)` instead of `self.fusion.run(candidates)`.
`FusionService` does **not** implement `__call__` (it doesn't inherit from `BaseService`, only has
`.run()`), so this call always raises a `TypeError`, regardless of which broker is used.

> **Note:** `Candidate.__repr__` formats `coordinates.ra`/`dec` assuming they are always numbers; if
> they are left as `None` (a candidate with no coordinates), printing the object raises a `TypeError`.
> That's why every simulated `Candidate` in this notebook is given explicit coordinates.


In [3]:
def simulated_fink_cone_search(ra, dec, radius_arcsec, **kwargs):
    """
    Replaces FinkBroker.cone_search(), which in production calls:
        POST https://api.ztf.fink-portal.org/api/v1/conesearch
    """
    c = Candidate(coordinates=Coordinates(ra=10.68, dec=41.27))
    c.add_broker(BrokerInfo(broker="Fink", object_id="ZTF23xxzzz"))
    c.classification = Classification(probabilities={"SN Ia": 0.55, "TDE": 0.45})
    return [c]

hunter.broker.cone_search = simulated_fink_cone_search  # tutorial-only override

try:
    hunter.cone_search(ra=10.68, dec=41.27, radius_arcsec=5.0)
except TypeError as exc:
    print("Reproduced bug in Hunter.cone_search():", exc)


Reproduced bug in Hunter.cone_search(): 'FusionService' object is not callable


**Workaround**, until the method is fixed (changing `self.fusion(candidates)` to
`self.fusion.run(candidates)` in `hunter.py` would be enough): call `broker.cone_search()` directly,
then `hunter.fusion.run()` and `hunter.process()` by hand.


In [4]:
raw = hunter.broker.cone_search(ra=10.68, dec=41.27, radius_arcsec=5.0)
fused = hunter.fusion.run(raw)
results = [hunter.process(c) for c in fused]

for r in results:
    print(r)


Candidate(object='ZTF23xxzzz', class='SN Ia (55.00%)', RA=10.68000, Dec=41.27000, brokers=[Fink])


## 3. `Hunter.get()` and `Hunter.classifications()`

`get()` downloads a **single object** by `object_id` and attaches its classification to the
`broker_classifications` dict. Note that it does **not** update `candidate.classification` and does
**not** run the candidate through `Hunter.process()` — the source code even has this behavior commented
out explicitly (`#return self.process(candidate)`), a sign that this is a deliberately "raw" path,
meant for quick inspection of a single object rather than ranking.


In [5]:
def simulated_get_object(object_id):
    """Replaces broker.get_object(), which in production queries the broker's API for a single object."""
    c = Candidate(coordinates=Coordinates(ra=150.324, dec=2.211))
    c.add_broker(BrokerInfo(broker="Fink", object_id=object_id))
    return c

def simulated_classifications(object_id):
    """Replaces broker.classifications()."""
    return Classification(probabilities={"SN Ia": 0.9})

hunter.broker.get_object = simulated_get_object
hunter.broker.classifications = simulated_classifications

candidate = hunter.get("ZTF23aabbcc")

print("candidate.classification (not yet populated):", candidate.classification)
print("candidate.broker_classifications:", candidate.broker_classifications)


candidate.classification (not yet populated): Classification(probabilities={}, classifier=None, confidence=None, calibrated=False, source=None, metadata={})
candidate.broker_classifications: {'Fink': Classification(probabilities={'SN Ia': 0.9}, classifier=None, confidence=None, calibrated=False, source=None, metadata={})}


In [6]:
# To get features/ranking on an object fetched via get(), you have to complete it by hand:
candidate.classification = candidate.broker_classifications["Fink"]
processed = hunter.process(candidate)
print("Total ranking:", processed.ranking.total)


Total ranking: 0.645


## 4. Practical note: `classifier` is required for Fink + `lsst` survey

`FinkBroker.CLASSIFIER_MAP` translates a THOR class name into the *tag*/*class* expected by the
broker's API, and the mapping depends on the survey. For `survey="ztf"`, the `/api/v1/latests` endpoint
also accepts `classifier=None` (all recent alerts); for `survey="lsst"`, the endpoint is `/api/v1/tags`
instead, where the **`tag` parameter is required** — if you leave `classifier=None`, the real network
call fails with a `BrokerError` listing the valid tags.


In [7]:
from thor.brokers.fink import FinkBroker
print(FinkBroker.CLASSIFIER_MAP)

# In production with survey="lsst":
#   hunter.survey = "lsst"
#   hunter.load("fink")
#   hunter.search(classifier="SN")   # -> translated into "most_likely_sn"
#   hunter.search()                  # -> classifier stays None -> BrokerError from the server


{'ztf': {'SN': 'SN', 'TDE': 'TDE'}, 'lsst': {'SN': 'most_likely_sn'}}


## Next notebook

**Tutorial 3** covers `Hunter.search_all()`: how to query several brokers together and get a single
list of candidates with classification **weighted** across brokers, via `FusionService`, plus the
newly implemented `agreement_score`.
